# 01 — Dense (Fully Connected) Network on MNIST

This notebook trains a classic **Multi-Layer Perceptron (MLP)** using only `Dense` layers on the MNIST dataset.

**Architecture:**
```
Input (28×28) → Flatten → Dense(256) → Dense(128) → Dense(10, softmax)
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TensorFlow version:', tf.__version__)

## 1. Load & Preprocess Data

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalise pixel values to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0

print(f'Train: {x_train.shape}, Test: {x_test.shape}')

# Preview some samples
fig, axes = plt.subplots(1, 8, figsize=(14, 2))
for i, ax in enumerate(axes):
    ax.imshow(x_train[i], cmap='gray')
    ax.set_title(str(y_train[i]))
    ax.axis('off')
plt.suptitle('Sample MNIST Images', y=1.02)
plt.tight_layout()
plt.show()

## 2. Build the Dense Model

In [ ]:
def build_dense_model():
    model = keras.Sequential([
        layers.Flatten(input_shape=(28, 28)),        # Flatten 28x28 → 784
        layers.Dense(256, activation='relu'),         # Hidden layer 1
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),         # Hidden layer 2
        layers.Dropout(0.3),
        layers.Dense(10, activation='softmax')        # Output: 10 digit classes
    ], name='Dense_MLP')
    return model

model = build_dense_model()
model.summary()

## 3. Compile & Train

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

start = time.time()
history = model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)
elapsed = time.time() - start
print(f'\nTotal training time: {elapsed:.1f}s')

## 4. Evaluate

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'Test Accuracy : {test_acc:.4f}')
print(f'Test Loss     : {test_loss:.4f}')
print(f'Total Params  : {model.count_params():,}')
print(f'Training Time : {elapsed:.1f}s')

## 5. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'],     label='Train')
ax1.plot(history.history['val_accuracy'], label='Validation')
ax1.set_title('Dense Model — Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(history.history['loss'],     label='Train')
ax2.plot(history.history['val_loss'], label='Validation')
ax2.set_title('Dense Model — Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## 6. Save Results

Save metrics to disk so the comparison notebook can load them.

In [ ]:
import json, os

os.makedirs('../results', exist_ok=True)
results = {
    'model': 'Dense MLP',
    'test_accuracy': float(test_acc),
    'test_loss': float(test_loss),
    'total_params': model.count_params(),
    'training_time_s': round(elapsed, 1),
    'history': {
        'accuracy':     history.history['accuracy'],
        'val_accuracy': history.history['val_accuracy'],
        'loss':         history.history['loss'],
        'val_loss':     history.history['val_loss'],
    }
}

with open('../results/dense_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Results saved to results/dense_results.json')